In [36]:
text = solve_exact_diagonalization(Lx = 1,Ly = 1, N_elec = 3, t1 = 1.0, t2 = -0.06742153978842166, U = 1.8084587278168796)
#print(text)

Lowest 3 energies:  [-1.99235590e+00 -1.99235590e+00 -1.65717901e-16]
Ground State Energy: -1.992355896690542
Observables:
{'double_occupancy': [0.16359729312752622, 0.1419236863699446, 0.13860503044338654], 'charge_correlation_matrix': [[1.3896553898615487, 0.9088706053771223, 0.8888564155808176], [0.9088706053771223, 1.2592798463489965, 0.758146969101203], [0.8888564155808176, 0.758146969101203, 1.23931678367117]], 'spin_correlation_matrix': [[0.5514496630135828, -0.08454376847480626, -0.1036830573191742], [-0.08454376847480626, 0.5186888256519134, -0.22867866675037657], [-0.1036830573191742, -0.22867866675037657, 0.5136724964232178]], 'spin_structure_factor': {'q=(0.00,0.00)': np.float64(0.24999999999999997), 'q=(3.14,0.00)': np.float64(0.5842995073461088), 'q=(0.00,3.14)': np.float64(0.6738108747671925), 'q=(2.09,2.09)': np.float64(0.5127297251414592)}}


In [35]:
from typing import TypedDict, Dict, Any, List, Optional
from langchain_core.tools import tool
import json
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import os
import h5py
from tenpy.networks import mps
from quspin.basis import spinful_fermion_basis_1d
from quspin.operators import hamiltonian

# Real-space 2D coordinates for a 3-site Kagome triangle motif
kagome_3site_positions = np.array([
    [0.0, 0.0],          # Site 0
    [0.5, np.sqrt(3)/2], # Site 1
    [1.0, 0.0]           # Site 2
])

# Set your base directory where the .h5 files live
BASE_DATABANK_DIR = "data_outputs/wavefunctions"

# Mapping site numbers to words for file naming
SITE_NUMBER_TO_WORD = {
    3: "three",
    6: "six",
    9: "nine",
    12: "twelve",
    15: "fifteen",
    30: "thirty",
    60: "sixty"
}

def resolve_h5_path(sites: int, t2: float, U: float, base_dir: str = BASE_DATABANK_DIR) -> str:
    """Generates the file path for a given set of model parameters."""
    site_word = SITE_NUMBER_TO_WORD.get(sites, f"{sites}")
    # Formats t2 and U to match string representation (e.g., three_site_kagome_t2_0.5_u_4.0.h5)
    filename = f"{site_word}_site_kagome_t2_{t2}_u_{U}.h5"
    return os.path.join(base_dir, filename)


def h5_to_dict(h5_group) -> Dict[str, Any]:
    """Recursively converts HDF5 groups and datasets into standard Python dictionaries and lists."""
    data = {}
    for key, item in h5_group.items():
        if (key == "psi"):
            if isinstance(item, h5py.Dataset): #ED numpy dataset
                val = item[()] #read dataset array/scalar
                val = val.tolist()
            if isinstance(item, h5py.Group): #TeNPy MPS/Subcluster Dict
                psi_type = item.attrs.get("psi_type", "tenpy_mps")
                if psi_type == "tenpy_mps":
                    val = thdf5.load(item, path = "mps_data")
                elif psi_type == "subcluster_dict":
                    sub_dict = {}
                    for key, val in item.items():
                        sub_dict[key] = val[()]
                        val = sub_dict
        elif isinstance(item, h5py.Dataset):
            val = item[()]  # Read dataset array/scalar
            if isinstance(val, np.ndarray):
                # Convert complex numbers to string/pairs or real values if applicable
                if np.iscomplexobj(val):
                    val = {"real": val.real.tolist(), "imag": val.imag.tolist()}
                else:
                    val = val.tolist()
            elif isinstance(val, (np.float32, np.float64, np.complex128)):
                val = float(val.real) if np.isreal(val) else {"real": float(val.real), "imag": float(val.imag)}
            elif isinstance(val, (np.int32, np.int64)):
                val = int(val)
            elif isinstance(val, bytes):
                val = val.decode("utf-8")
            data[key] = val
        elif isinstance(item, h5py.Group):
            data[key] = h5_to_dict(item)
    return data

def dict_to_h5(h5_group, data_dict: Dict[str, Any]):
    """Recursively writes nested Python dictionaries back into an HDF5 group/file structure."""
    for key, val in data_dict.items():
        if (key == "psi"):
            if isinstance(val, MPS): #TeNPy MPS object
                psi_subgroup = h5_group.create_group("psi")
                psi_subgroup.attrs["psi_type"] = "tenpy_mps"
                thdf5.save(psi, psi_subgroup, path="mps_data")
            elif isinstance(val, np.ndarray): #ED NumPy array
                ds = h5_group.create_dataset("psi", data=val)
                ds.attrs["psi_type"] = "numpy_ndarray"
            elif isinstance(val, dict): #subcluster waveforms
                psi_subgroup = h5_group.create_group("psi")
                psi_subgroup.attrs["psi_type"] = "subcluster_dict"
                for cluster_id, sub_psi in psi.items():
                    psi_subgroup.create_dataset(str(cluster_id), data=sub_psi)
                            
        elif isinstance(val, dict):
            subgroup = h5_group.create_group(key)
            dict_to_h5(subgroup, val)
            
        else:
            # Handle complex number dictionary structures if passed back
            if isinstance(val, dict) and "real" in val and "imag" in val:
                val = np.array(val["real"]) + 1j * np.array(val["imag"])
            
            val_arr = np.array(val)
            h5_group.create_dataset(key, data=val_arr)

def get_kagome_t1_bonds(nx: int, ny: int) -> list:
    """
    Calculates all nearest-neighbor (t1) bonds for an nx by ny lattice.
    Distance ~ 1.0. 
    Returns a list of tuples (i, j) where i < j for herm_con=True.
    """
    positions = get_kagome_positions(nx, ny)
    num_sites = len(positions)
    t1_bonds = []
    
    for i in range(num_sites):
        # Starting j at i+1 strictly enforces i < j
        for j in range(i + 1, num_sites):
            
            dist = np.linalg.norm(positions[i] - positions[j])
            
            # Check if distance is exactly 1.0 (with small float tolerance)
            if np.isclose(dist, 1.0, atol=1e-3):
                t1_bonds.append((i, j))
                
    return t1_bonds


def get_kagome_t2_bonds(nx: int, ny: int) -> list:
    """
    Calculates all next-nearest-neighbor (t2) bonds for an nx by ny lattice.
    Distance ~ sqrt(3) (approx 1.732). 
    Returns a list of tuples (i, j) where i < j for herm_con=True.
    """
    positions = get_kagome_positions(nx, ny)
    num_sites = len(positions)
    t2_bonds = []
    
    target_dist = np.sqrt(3)
    
    for i in range(num_sites):
        # Starting j at i+1 strictly enforces i < j
        for j in range(i + 1, num_sites):
            
            dist = np.linalg.norm(positions[i] - positions[j])
            
            # Check if distance is exactly sqrt(3) (with small float tolerance)
            if np.isclose(dist, target_dist, atol=1e-3):
                t2_bonds.append((i, j))
                
    return t2_bonds

        
def build_fermi_hubbard_hamiltonian(Lx: int, Ly: int, N_up: int, N_down: int, t1: float, t2: float, U: float):
    """
    Builds a sparse SciPy matrix for the Fermi-Hubbard model on a Kagome sub-cluster.
    Restricts the Hilbert space to exactly N_up and N_down electrons.
    """
    
    # 1. Restrict the Hilbert Space
    # This single line drops the matrix size from 4^N down to (N C N_up) * (N C N_down)
    sites = 3 * Lx * Ly
    basis = spinful_fermion_basis_1d(L=sites, Nf=(N_up, N_down))

    # 2. Get Lattice Bonds (You will need a helper function for your specific geometry)
    # These should be lists of tuples representing directional bonds: [(i, j), ...]
    # For Kagome, you only want to list each bond ONCE (e.g., i < j) because we use herm_con=True later.
    t1_bonds = get_kagome_t1_bonds(Lx, Ly)  # Nearest-neighbor bonds
    t2_bonds = get_kagome_t2_bonds(Lx, Ly)  # Next-nearest-neighbor bonds
    #t1_bonds, t2_bonds = get_kagome_bonds(site

    # 3. Format Interaction Lists for QuSpin
    # QuSpin uses operator strings where the left side of '|' is spin-up, right side is spin-down.
    
    # Hubbard U: n_{i,up} * n_{i,down}
    interaction_U = [[U, i, i] for i in range(sites)]

    # Hopping t: c^\dagger_i c_j 
    # '-' sign is included here because the physical hopping lowers the energy
    hop_up_t1 = [[-t1, i, j] for i, j in t1_bonds] + [[-t1, j, i] for i, j in t1_bonds]
    hop_dn_t1 = [[-t1, i, j] for i, j in t1_bonds] + [[-t1, j, i] for i, j in t1_bonds]
    
    hop_up_t2 = [[-t2, i, j] for i, j in t2_bonds] + [[-t2, j, i] for i, j in t2_bonds]
    hop_dn_t2 = [[-t2, i, j] for i, j in t2_bonds] + [[-t2, j, i] for i, j in t2_bonds]

    # Combine into QuSpin's static operator list
    static_operators = [
        ["n|n", interaction_U],     # U on both spins
        ["+-|", hop_up_t1],         # t1 hopping for spin-up
        ["|+-", hop_dn_t1],         # t1 hopping for spin-down
        ["+-|", hop_up_t2],         # t2 hopping for spin-up
        ["|+-", hop_dn_t2],         # t2 hopping for spin-down
    ]

    # 4. Build the Hamiltonian
    # FALSE herm_con=True automatically generates the Hermitian conjugates (the c^\dagger_j c_i terms), FALSE
    # meaning you don't have to manually write the reverse bonds.
    H = hamiltonian(
        static_operators, 
        dynamic_list=[], 
        basis=basis, 
        dtype=np.complex128, 
        check_herm=False,
        check_pcon=False, # Skips a slow internal symmetry check
        check_symm=False  
    )

    # 5. Extract and return as standard SciPy CSR sparse matrix
    return H.tocsr()

    
# 1. State Schema shared across the LangGraph pipeline
class PipelineState(TypedDict):
    lattice_params: Dict[str, Any]  # e.g., {"sites": 12, "U": 4.0, "t": 1.0, "filling": 0.5}
    nqs_predictions: Dict[str, Any] # Matrices from NQS framework
    oracle_results: Dict[str, Any]  # Verified exact results
    messages: List[Any]             # Agent conversational history


# 2. Tooling for the LLM Oracle
@tool
def query_databank(lattice_params: str) -> str:
    """
    Queries the databank for an existing .h5 file matching system parameters.
    Input should be a JSON string with keys: 'sites', 't2', 'U'.
    Example: '{"sites": 3, "t2": 0.5, "U": 4.0}'
    """
    try:
        params = json.loads(lattice_params)
        sites = int(params["sites"])
        t2 = float(params["t2"])
        U = float(params["U"])
    except (json.JSONDecodeError, KeyError) as e:
        return json.dumps({"status": "error", "message": f"Invalid input parameters format: {str(e)}"})

    filepath = resolve_h5_path(sites, t2, U)

    if not os.path.exists(filepath):
        return json.dumps({
            "status": "miss",
            "message": f"No exact dataset found at target path: {filepath}"
        })

    try:
        with h5py.File(filepath, "r") as h5_file:
            extracted_data = h5_to_dict(h5_file)
            
        return json.dumps({
            "status": "hit",
            "filepath": filepath,
            "data": extracted_data
        })
    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"Failed to read HDF5 file at {filepath}: {str(e)}"
        })


@tool
def save_to_databank(payload: str) -> str:
    """
    Saves newly computed ground state wavefunctions and observables to an .h5 file in the databank.
    Input should be a JSON string with 'lattice_params' and 'results_data'.
    
    Expected structure of 'results_data':
    {
        'psi': [...],
        'model_params': {...},
        'energy': -1.23,
        'average_energy': -1.21,  # Twisted Boundary Condition average
        'observables': {
            'charge_density': [...],
            'double_occupancy': [...],
            'spin_corr': [...],
            'charge_corr': [...],
            'q_vectors': [...],
            'S_q_spin': [...],
            'S_q_charge': [...]
        },
        'average_observables': {...}
    }
    """
    try:
        input_data = json.loads(payload)
        params = input_data["lattice_params"]
        results_data = input_data["results_data"]
        
        sites = int(params["sites"])
        t2 = float(params["t2"])
        U = float(params["U"])
    except (json.JSONDecodeError, KeyError) as e:
        return json.dumps({"status": "error", "message": f"Invalid input structure: {str(e)}"})

    filepath = resolve_h5_path(sites, t2, U)
    
    # Ensure folder structure exists
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    try:
        with h5py.File(filepath, "w") as h5_file:
            dict_to_h5(h5_file, results_data)

        return json.dumps({
            "status": "success",
            "message": f"Successfully cached observables to {filepath}"
        })
    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"Failed to write HDF5 file at {filepath}: {str(e)}"
        })


#@tool
def solve_exact_diagonalization(Lx : int = 1, Ly : int = 1, N_elec : int = 3, t1: float = 1.0, t2: float = 1.0, U: float = 4.0) -> str:
    """
    Solves exact quantum properties for small Kagome sub-clusters using SciPy sparse ED.
    Expects number of L_x (int), L_y (int), N_elec (int), t1 (float), t2 (float), and U (float)
    """
    # Step A: Parse input JSON string from Agent
    #changed inputs to directly get sites, t2, and U
    N_up = N_elec // 2 + (N_elec % 2)
    N_down = N_elec // 2
    sites = 3* Lx * Ly

    # Step B: Build your sparse Kagome Hamiltonian H
    # (Note: Replace this stub with your actual Hamiltonian builder)
    num_qubits = 2 * sites
    #H_sparse = sp.csr_matrix((2**num_qubits, 2**num_qubits), dtype=np.complex128)
    H_sparse = build_fermi_hubbard_hamiltonian(Lx = Lx, Ly = Ly, N_up = N_up, N_down = N_down, t1=t1, t2 = t2, U=U)
    basis = spinful_fermion_basis_1d(L=sites, Nf=(N_up, N_down))

    # Step C: Diagonalize to find Ground State (k=1 smallest algebraic eigenvalue)
    k_request = min(4, H_sparse.shape[0] - 2)
    ground_eigenvalue, ground_eigenvector = spla.eigsh(
        #H_sparse, k=1, which="SA"
        #H_sparse, k=3, which="SA"
        H_sparse, k = k_request, which = 'SA'
    )
    
    # Flatten state vector for 1D array operations
    ground_state = ground_eigenvector[:, 0]
    e_ground = float(np.real(ground_eigenvalue[0]))
    tolerance = 1e-8
    degenerate_indices = np.where(np.abs(eigenvalues - ground_energy) < tolerance)[0]
    degeneracy_count = len(degenerate_indices)

    # Step D: Get site geometry coordinates
    site_positions = get_kagome_positions(Lx, Ly)

    # Step E: Compute Observables via the helper function, NOTE: for all degeneracy states
    accumulated_observables = {
        'double_occupancy': np.zeros(sites),
        'charge_correlation_matrix': np.zeroes((sites, sites)),
        'spin_correlation_matrix': np.zeros((sites, sites))
    }
    accumlated_s_sq = {}
    accumlated_c_sq = {}

    for idx in degenerate_indices:
        psi_n = ground_eigenvector[:, idx]
        obs_n = = compute_kagome_observables(
            ground_state=ground_state,
            num_sites=sites,
            site_positions=site_positions,
            basis = basis
        )
        accumulated_observables['double_occupancy'] += np.array(obs_n["double_occupancy"])
        accumulated_observables['charge_correlation_matrix'] += np.array(obs_n["charge_correlation_matrix"])
        accumulated_observables['spin_correlation_matrix'] += np.array(obs_n["spin_correlation_matrix"])

        for q_key, sq_val in obs_n["spin_structure_factor"].items()
            accumulated_s_sq[q_key] = accumulated_s_sq.get(q_key, 0.0) + sq_val 
        for q_key, sq_val in obs_n["charge_structure_factor"].items()
            accumulated_c_sq[q_key] = accumulated_c_sq.get(q_key, 0.0) + sq_val 

    observables = {
        key: (val / degeneracy_count).tolist() for key, val in accumulated_observables.items()
    }
    observables["spin_structure_factor"] = {
        q_key: sq_val / degeneracy_count for q_key, sq_val in accumulated_s_sq.items()
    }
    observables["charge_structure_factor"] = {
        q_key: sq_val / degeneracy_count for q_key, sq_val in accumulated_c_sq.items()
    }
    observables["degeneracy"] = int(degeneracy_count)

    # Step F: Assemble and Return Full Results Dict as JSON String
    output_payload = {
        "sites": sites,
        "ground_state_energy": e_ground,
        **observables  # Unpacks double_occupancy, charge/spin correlation matrices, S(q), C S(q), degeneracy*
    }
    print(f"Ground State Energy: {e_ground}\nObservables:\n{observables}")

    return json.dumps(output_payload)


@tool
def solve_mps_tenpy(cluster_params: str) -> str:
    """Runs Matrix Product States (MPS/DMRG) via TeNPy for quasi-1D strips or larger systems (N > 12)."""
    params = json.loads(cluster_params)
    # Run TeNPy DMRG simulation...
    return json.dumps({"status": "completed", "method": "TeNPy DMRG", "energy": -2.3456})


@tool
def extrapolate_from_subclusters(subcluster_results_list: str) -> str:
    """Combines/extrapolates exact observable matrices from multiple smaller sub-clusters 
    (e.g., aggregating 3x3 or 6x6 Kagome motifs to reconstruct an NxN prediction)."""
    results = json.loads(subcluster_results_list)
    # Aggregation algorithm (e.g., Cluster Perturbation Theory / Matrix Stitching)
    return json.dumps({"status": "extrapolated", "reconstructed_matrix_shape": [15, 15]})

def build_sparse_qubit_operator(target_qubit: int, single_qubit_op: sp.csr_matrix, num_qubits: int) -> sp.csr_matrix:
    """Builds an efficient N-qubit sparse Kronecker product operator."""
    I2 = sp.eye(2, format='csr', dtype=np.complex128)
    
    op_list = [single_qubit_op if i == target_qubit else I2 for i in range(num_qubits)]
    
    # Efficient sparse Kronecker chain
    result = op_list[0]
    for op in op_list[1:]:
        result = sp.kron(result, op, format='csr')
    return result

def compute_kagome_observables(ground_state: np.ndarray, num_sites: int, site_positions: np.ndarray, basis):
    """
    Computes Ground State Energy, Double Occupancy, Charge Correlation,
    Spin-Spin Correlation, and 2D Structure Factors for Kagome clusters
    within the constrained particle-conserving basis.
    
    Parameters:
        ground_state: Ground state eigenvector (dimension matches the constrained basis)
        num_sites: N (e.g., 3, 6, 9, 12)
        site_positions: Nx2 array of (x, y) coordinates for Kagome sites
        basis: The exact QuSpin basis object used to build the Hamiltonian
    """
    
    # 1 & 2. Build Sparse Number Operators n_i_up and n_i_down using QuSpin
    # This replaces the raw Pauli matrix approach and forces the operators 
    # to match the constrained dimension of the ground_state.
    n_up = []
    n_dn = []
    
    for i in range(num_sites):
        # Create constrained operators directly via QuSpin
        # "n|" means particle number operator for spin-up
        # "|n" means particle number operator for spin-down
        op_up = hamiltonian([["n|", [[1.0, i]]]], [], basis=basis, dtype=np.complex128, 
                            check_herm=False, check_symm=False, check_pcon=False)
        op_dn = hamiltonian([["|n", [[1.0, i]]]], [], basis=basis, dtype=np.complex128, 
                            check_herm=False, check_symm=False, check_pcon=False)
        
        # Convert to scipy sparse format so matrix math (@, +) works identically
        n_up.append(op_up.tocsr())
        n_dn.append(op_dn.tocsr())

    # 3. Double Occupancy Vector: <n_{i,up} * n_{i,dn}>
    double_occ = []
    for i in range(num_sites):
        D_op = n_up[i] @ n_dn[i]
        d_val = np.real(np.vdot(ground_state, D_op.dot(ground_state)))
        double_occ.append(float(d_val))

    # 4. Total Site Charge Operators N_i = n_{i,up} + n_{i,dn}
    N_op = [n_up[i] + n_dn[i] for i in range(num_sites)]
    
    # Charge Correlation Matrix (NxN)
    charge_corr = np.zeros((num_sites, num_sites))
    for i in range(num_sites):
        for j in range(num_sites):
            C_ij = N_op[i] @ N_op[j]
            charge_corr[i, j] = np.real(np.vdot(ground_state, C_ij.dot(ground_state)))

    # 5. Spin-Spin Correlation Matrix S_i . S_j = 3 * S^z_i S^z_j
    spin_corr = np.zeros((num_sites, num_sites))
    for i in range(num_sites):
        S_z_i = 0.5 * (n_up[i] - n_dn[i])
        for j in range(num_sites):
            S_z_j = 0.5 * (n_up[j] - n_dn[j])
            S_ij = 3.0 * (S_z_i @ S_z_j)  # Isotropic multiplier for spin singlet
            spin_corr[i, j] = np.real(np.vdot(ground_state, S_ij.dot(ground_state)))

    # 6. 2D Spin Structure Factor S(q)
    # Example q-points in 2D Brillouin Zone
    q_points = [
        np.array([0.0, 0.0]),
        np.array([np.pi, 0.0]),
        np.array([0.0, np.pi]),
        np.array([2*np.pi/3, 2*np.pi/3])
    ]
    
    spin_structure_factor = {}
    for q in q_points:
        sq_val = 0.0
        for j in range(num_sites):
            for k in range(num_sites):
                r_jk = site_positions[j] - site_positions[k]
                phase = np.exp(-1j * np.dot(q, r_jk))
                sq_val += phase * spin_corr[j, k]
        spin_structure_factor[f"q=({q[0]:.2f},{q[1]:.2f})"] = np.real(sq_val) / num_sites
    #7. 2D Charge Structure Factor S(q), same as spin but for charge
    charge_structure_factor = {}
    for q in q_points:
        c_sq_val = 0.0
        for j in range(num_sites):
            for k in range(num_sites):
                r_jk = site_positions[j] - site_positions[k]
                phase = np.exp(-1j * np.dot(q, r_jk))
                c_sq_val += phase * spin_corr[j, k]
        charge_structure_factor[f"q=({q[0]:.2f},{q[1]:.2f})"] = np.real(c_sq_val) / num_sites
    return {
        "double_occupancy": double_occ,
        "charge_correlation_matrix": charge_corr.tolist(),
        "spin_correlation_matrix": spin_corr.tolist(),
        "spin_structure_factor": spin_structure_factor,
        "charge_structure_factor": charge_structure_factor
    }

#Helper: Generate 2D real-space coordinates for Kagome clusters
def get_kagome_positions(nx: int, ny: int) -> np.ndarray:
    """
    Generates (x, y) coordinates for an nx by ny Kagome lattice.
    Each unit cell contains 3 sites (base=0,1; tip=2).
    Total sites generated = nx * ny * 3.
    """
    positions = []
    
    for y in range(ny):
        for x in range(nx):
            # Kagome Bravais lattice shift vectors.
            # Moving up one unit in 'y' shifts the cell diagonally.
            x_shift = (x * 2.0) + (y * 1.0)
            y_shift = (y * np.sqrt(3))
            
            # Site 0: Bottom left of the unit cell triangle
            positions.append([x_shift + 0.0, y_shift + 0.0])
            
            # Site 1: Bottom right of the unit cell triangle
            positions.append([x_shift + 1.0, y_shift + 0.0])
            
            # Site 2: Top tip of the unit cell triangle
            positions.append([x_shift + 0.5, y_shift + np.sqrt(3)/2])
            
    return np.array(positions)
    return np.array(positions)

SyntaxError: invalid syntax (3011181906.py, line 364)

In [ ]:
import os
import json
from typing import TypedDict, Dict, Any, List
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver  # In-memory checkpointer
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, trim_messages
from tools import (
    PipelineState, 
    query_databank, 
    save_to_databank, 
    solve_exact_diagonalization, 
    solve_mps_tenpy, 
    extrapolate_from_subclusters
)
import uuid

# Define tools list
tools = [
    query_databank, 
    save_to_databank, 
    solve_exact_diagonalization, 
    solve_mps_tenpy, 
    extrapolate_from_subclusters
]

def load_oracle_rules(filepath: str = "oracle_rules.md") -> str:
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            return f.read()
    return "No external context file found."

ORACLE_RULES = load_oracle_rules()

# Initialize LLM with tool-calling capabilities
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)
#LLM API

def run_verification_job(sites: int, U: float, t: float):
    # Unique thread ID per job prevents state cross-contamination
    job_id = f"job_N{sites}_U{U}_{uuid.uuid4().hex[:6]}"
    config = {"configurable": {"thread_id": job_id}}
    
    initial_state = {
        "lattice_params": {"sites": sites, "U": U, "t": t, "filling": 0.5},
        "nqs_predictions": {},
        "oracle_results": {},
        "messages": [HumanMessage(content=f"Verify NQS results for {sites}-site Kagome lattice.")]
    }
    
    # Execute graph stream
    for event in app.stream(initial_state, config=config):
        print(f"--- Step Executed in Thread {job_id} ---")

# Run separate jobs independently
run_verification_job(sites=6, U=4.0, t=1.0)   # Job 1
run_verification_job(sites=15, U=4.0, t=1.0)  # Job 2

# Node 1: NQS Interface Stub (Hook for external team)
def nqs_predictor_node(state: PipelineState) -> Dict[str, Any]:
    """Receives system parameters and acquires matrix predictions from external NQS network."""
    params = state["lattice_params"]
    n = params["sites"]
    
    # Mock NQS output payload matching your observables list
    nqs_out = {
        "ground_state_energy": -1.21 * n,
        "spin_spin_correlation": np.random.rand(n, n).tolist(),
        "charge_charge_correlation": np.random.rand(n, n).tolist(),
        "double_occupancy": 0.13,
        "spin_structure_factor": [0.5, 0.2, 0.1],
        "charge_structure_factor": [0.4, 0.3, 0.1]
    }
    return {"nqs_predictions": nqs_out}


# Node 2: LLM Oracle Agent
def oracle_agent_node(state: PipelineState) -> Dict[str, Any]:
    messages = state.get("messages", [])
    
    # Message Trimming: Keep history within safe token limits (prevents context overload)
    trimmed_history = trim_messages(
        messages,
        max_tokens=4000,
        strategy="last",
        token_counter=llm,
        start_on="human"
    )
    
    # Inject loaded rules context + system prompt
    system_prompt = SystemMessage(
        content=f"""
        You are the Exact Verification Oracle Agent.
        
        === EXTERNAL PHYSICS RULES & CONTEXT ===
        {PHYSICS_RULES}
        ========================================
        
        Current Target Params: {json.dumps(state['lattice_params'])}
        NQS Predictions: {json.dumps(state['nqs_predictions'])}
        """
    )
    
    response = llm.invoke([system_prompt] + trimmed_history)
    return {"messages": messages + [response]}


# Build Graph Structure
workflow = StateGraph(PipelineState)

# Add Nodes
workflow.add_node("nqs_predictor", nqs_predictor_node)
workflow.add_node("oracle_agent", oracle_agent_node)
workflow.add_node("tools", ToolNode(tools))

# Add Edges
workflow.add_edge(START, "nqs_predictor")
workflow.add_edge("nqs_predictor", "oracle_agent")

# Dynamic Tool Execution Edges
workflow.add_conditional_edges(
    "oracle_agent",
    tools_condition,  # Directs to "tools" if LLM generated tool calls, else END
)
workflow.add_edge("tools", "oracle_agent") # Loop tool execution back to agent

# Enable Checkpointer Memory
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)